# EEVR Preprocessing and Alignment

Builds the model-ready EDA, PPG and TEXT files from the raw EEVR files.

**Inputs** (`EEVR/Data_files/`, read-only)
- `Textdata.csv`: free-text description per participant and video
- `VADS.csv`: valence / arousal / dominance self-reports
- `EDA_labels.csv`, `PPG_labels.csv`: extracted physiological features

**Outputs** (`EEVR/Data_files/`)
- `EDA_preprocessed_aligned.csv`, `PPG_preprocessed_aligned.csv`, `TEXT_preprocessed_aligned.csv`: row-aligned by `Data ID`
- `text_data_labels.csv`: same content as `TEXT_preprocessed_aligned.csv`, kept for older notebooks

**No feature scaling is done here.** Fitting a scaler on the whole dataset leaks statistics of the held-out
participant into training. Scaling has to be fit on the training fold inside each experiment notebook.

## 1. Environment setup

In [14]:
import os
os.chdir("/notebooks")

In [ ]:
from lib.install import prepare_env
_ = prepare_env()

## 2. Imports

In [16]:
import string
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

for package in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(package, quiet=True)                              # NLTK resources for text cleaning

## 3. Configuration

In [17]:
DATA_DIR = Path("EEVR/Data_files")

INPUT_CONFIG = {
    "text_path": DATA_DIR / "Textdata.csv",
    "vads_path": DATA_DIR / "VADS.csv",
    "eda_path":  DATA_DIR / "EDA_labels.csv",
    "ppg_path":  DATA_DIR / "PPG_labels.csv",
}

OUTPUT_CONFIG = {
    "eda_path":         DATA_DIR / "EDA_preprocessed_aligned.csv",
    "ppg_path":         DATA_DIR / "PPG_preprocessed_aligned.csv",
    "text_path":        DATA_DIR / "TEXT_preprocessed_aligned.csv",
    "text_labels_path": DATA_DIR / "text_data_labels.csv",
    "table_dir":        Path("output/tables"),
}

BASELINE_VIDEO_ID = 0                                                # baseline recording, has no text description

identifiers = [
    "Participant ID",
    "Video ID",
    "arousal_category",
    "valence_category",
    "taskwiselabel",
]

eda_features = [
    "ku_eda", "sk_eda", "dynrange", "slope", "variance",
    "entropy", "insc", "fd_mean", "max_scr", "min_scr",
    "nSCR", "meanAmpSCR", "meanRespSCR", "sumAmpSCR", "sumRespSCR",
]

ppg_features = [
    "BPM", "IBI", "PPG_Rate_Mean", "HRV_MedianNN", "HRV_Prc20NN",
    "HRV_MinNN", "HRV_HTI", "HRV_TINN", "HRV_LF", "HRV_VHF",
    "HRV_LFn", "HRV_HFn", "HRV_LnHF", "HRV_SD1SD2", "HRV_CVI",
    "HRV_PSS", "HRV_PAS", "HRV_PI", "HRV_C1d", "HRV_C1a",
    "HRV_DFA_alpha1", "HRV_MFDFA_alpha1_Width", "HRV_MFDFA_alpha1_Peak",
    "HRV_MFDFA_alpha1_Mean", "HRV_MFDFA_alpha1_Max",
    "HRV_MFDFA_alpha1_Delta", "HRV_MFDFA_alpha1_Asymmetry",
    "HRV_ApEn", "HRV_ShanEn", "HRV_FuzzyEn", "HRV_MSEn",
    "HRV_CMSEn", "HRV_RCMSEn", "HRV_CD", "HRV_HFD", "HRV_KFD", "HRV_LZC",
]

text_features = [
    "Text Description",
]

## 4. Utility functions

In [18]:
# ============================================================
# Utility: CSV read, drops pandas index columns
# ============================================================
def csv_read(path):
    df = pd.read_csv(path)
    unnamed_cols = [c for c in df.columns if str(c).startswith("Unnamed")]
    return df.drop(columns=unnamed_cols)


# ============================================================
# Utility: Video ID to integer
# Handles values like V0, V1, v2, 1, 1.0
# ============================================================
def clean_video_id(df):
    df = df.copy()

    video_id_clean = (
        df["Video ID"]
        .astype(str)
        .str.strip()
        .str.replace(r"^[Vv]", "", regex=True)
    )
    video_id_clean = pd.to_numeric(video_id_clean, errors="coerce")

    if video_id_clean.isna().any():
        display(df.loc[video_id_clean.isna(), ["Video ID"]].drop_duplicates())
        raise ValueError("Some Video ID values could not be converted to integer.")

    df["Video ID"] = video_id_clean.astype(int)
    return df


# ============================================================
# Utility: Data ID = 1000 * Participant ID + Video ID
# Unique key used to align rows across modalities
# ============================================================
def add_dataid(df):
    df = df.copy()
    data_id = 1000 * df["Participant ID"].astype(int) + df["Video ID"].astype(int)

    if "Data ID" in df.columns:
        if not (df["Data ID"].astype(int) == data_id).all():
            raise ValueError("Existing Data ID column does not match Participant ID / Video ID.")
        return df

    df.insert(0, "Data ID", data_id)
    return df


# ============================================================
# Utility: drop rows with missing values in the given columns
# ============================================================
def drop_missing(df, columns, name):
    missing = df[columns].isna().any(axis=1)
    print(f"{name}: {missing.sum()} rows with missing values in {len(columns)} required columns")
    return df.loc[~missing].reset_index(drop=True)


# ============================================================
# Utility: drop baseline recording
# ============================================================
def remove_baseline(df, name):
    before = len(df)
    df = df[df["Video ID"] != BASELINE_VIDEO_ID].reset_index(drop=True)
    print(f"{name}: removed {before - len(df)} baseline rows (Video ID {BASELINE_VIDEO_ID})")
    return df


# ============================================================
# Utility: escape text for LaTeX tables
# ============================================================
def latex_escape(x):
    if pd.isna(x):
        return ""

    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    return "".join(replacements.get(ch, ch) for ch in str(x))

## 5. Text and labels

Join the text descriptions with the self-reports and build the three binary targets:
- `arousal_category`, `valence_category`: rating 1, 2, 3 -> 0 and 4, 5 -> 1
- `taskwiselabel` (stimulus label): CMA starting with `LV` -> 0, `HV` -> 1

In [19]:
text_df = clean_video_id(csv_read(INPUT_CONFIG["text_path"]))
vads_df = clean_video_id(csv_read(INPUT_CONFIG["vads_path"]))

join_cols = [
    "Participant ID",
    "Playlist ID",
    "Video ID",
    "CMA",
    "Arousal",
    "Valence",
]

text_df = text_df.merge(vads_df, on=join_cols, how="inner")

# ============================================================
# Arousal and valence categories
# ============================================================
text_df["Arousal"] = pd.to_numeric(text_df["Arousal"], errors="coerce")
text_df["Valence"] = pd.to_numeric(text_df["Valence"], errors="coerce")
text_df = text_df.dropna(subset=["Arousal", "Valence"]).copy()

text_df["Arousal"] = text_df["Arousal"].astype(int)
text_df["Valence"] = text_df["Valence"].astype(int)

text_df["arousal_category"] = (text_df["Arousal"] >= 4).astype(int)  # 1,2,3 -> 0 | 4,5 -> 1
text_df["valence_category"] = (text_df["Valence"] >= 4).astype(int)

# ============================================================
# Stimulus label from CMA
# ============================================================
text_df["CMA"] = text_df["CMA"].astype(str).str.strip()

text_df["taskwiselabel"] = np.where(
    text_df["CMA"].str.startswith("LV"),
    0,
    np.where(text_df["CMA"].str.startswith("HV"), 1, np.nan),
)
text_df = text_df.dropna(subset=["taskwiselabel"]).copy()           # drop baseline / non LV-HV rows
text_df["taskwiselabel"] = text_df["taskwiselabel"].astype(int)

print("Text + labels shape:", text_df.shape)
display(text_df.head())

Text + labels shape: (296, 13)


,Text Description,Participant ID,Playlist ID,Video ID,Segment Name,CMA,Valence,Arousal,Dominance,significance,arousal_category,valence_category,taskwiselabel
0,In this video childrens were shown whoes homes...,1,1,1,TheDisplaced,LVLA,4,3,2,3,0,1,0
1,In this video people and childrens were shown ...,1,1,2,HappyLand,LVLA,3,4,4,3,1,0,0
2,In this video a person who was confined in jai...,1,1,3,Jailbreak,LVHA,3,3,3,2,0,0,0
3,This video was about war and it impact all nat...,1,1,4,Warknownonation,LVHA,2,2,4,1,0,0,0
4,In this video I jumped into a canyon using a r...,1,1,5,CanyonSwing,HVHA,3,3,3,4,0,0,1


### 5.1 Valence category vs. stimulus label

Examples where the participant's reported valence agrees / disagrees with the intended stimulus label.
Saved as a LaTeX table for the thesis.

In [20]:
cols_to_show = [
    "Participant ID",
    "Video ID",
    "CMA",
    "Arousal",
    "Valence",
    "arousal_category",
    "valence_category",
    "taskwiselabel",
]

is_match = text_df["valence_category"].eq(text_df["taskwiselabel"])

examples_df = (
    pd.concat([
        text_df.loc[is_match, cols_to_show].drop_duplicates().head(10),
        text_df.loc[~is_match, cols_to_show].drop_duplicates().head(10),
    ], ignore_index=True)
    .sort_values(by=["Participant ID", "Video ID"])
    .reset_index(drop=True)
)

display(examples_df)

# ============================================================
# Store LaTeX table
# ============================================================
table_dir = OUTPUT_CONFIG["table_dir"]
table_dir.mkdir(parents=True, exist_ok=True)
latex_path = table_dir / "valence_stimulus_match_examples.tex"

with open(latex_path, "w", encoding="utf-8") as f:
    f.write("\\begin{table}[htbp]\n")
    f.write("\\centering\n")
    f.write("\\scriptsize\n")
    f.write("\\caption{Examples comparing participant-reported valence category and stimulus label. Rows are sorted by participant ID and video ID.}\n")
    f.write("\\label{tab:valence_stimulus_match_examples}\n")
    f.write("\\begin{tabular}{lllccccc}\n")
    f.write("\\hline\n")
    f.write("\\rowcolor{gray!15}\n")
    f.write("\\textbf{Participant ID} & \\textbf{Video ID} & \\textbf{CMA} & \\textbf{Arousal} & \\textbf{Valence} & \\shortstack{\\textbf{Arousal}\\\\\\textbf{category}} & \\shortstack{\\textbf{Valence}\\\\\\textbf{category}} & \\shortstack{\\textbf{Stimulus}\\\\\\textbf{label}} \\\\\n")
    f.write("\\hline\n")

    for _, row in examples_df.iterrows():
        f.write(" & ".join(latex_escape(row[c]) for c in cols_to_show) + " \\\\\n")

    f.write("\\hline\n")
    f.write("\\end{tabular}\n")
    f.write("\\end{table}\n")

print("Saved LaTeX table:", latex_path)

,Participant ID,Video ID,CMA,Arousal,Valence,arousal_category,valence_category,taskwiselabel
0,1,1,LVLA,3,4,0,1,0
1,1,2,LVLA,4,3,1,0,0
2,1,3,LVHA,3,3,0,0,0
3,1,4,LVHA,2,2,0,0,0
4,1,5,HVHA,3,3,0,0,1
5,1,6,HVLA,3,3,0,0,1
6,1,7,HVHA,4,4,1,1,1
7,1,8,HVLA,2,4,0,1,1
8,2,1,LVLA,3,2,0,0,0
9,2,2,LVLA,3,1,0,0,0


Saved LaTeX table: output/tables/valence_stimulus_match_examples.tex


## 6. EDA preprocessing

1. Remove rows with missing EDA features or labels
2. Remove the baseline recording (Video ID 0), it has no text description

In [21]:
eda_df = add_dataid(clean_video_id(csv_read(INPUT_CONFIG["eda_path"])))
print("EDA raw shape:", eda_df.shape)

eda_df = drop_missing(eda_df, eda_features + identifiers, "EDA")
eda_df = remove_baseline(eda_df, "EDA")

print("EDA clean shape:", eda_df.shape)

EDA raw shape: (333, 29)
EDA: 0 rows with missing values in 20 required columns
EDA: removed 37 baseline rows (Video ID 0)
EDA clean shape: (296, 29)


## 7. PPG preprocessing

Same steps as EDA.

In [22]:
ppg_df = add_dataid(clean_video_id(csv_read(INPUT_CONFIG["ppg_path"])))
print("PPG raw shape:", ppg_df.shape)

ppg_df = drop_missing(ppg_df, ppg_features + identifiers, "PPG")
ppg_df = remove_baseline(ppg_df, "PPG")

print("PPG clean shape:", ppg_df.shape)

PPG raw shape: (333, 53)
PPG: 0 rows with missing values in 42 required columns
PPG: removed 37 baseline rows (Video ID 0)
PPG clean shape: (296, 53)


## 8. TEXT preprocessing

1. Remove rows with missing text or labels
2. Clean the description with NLTK: lowercase, tokenize, strip punctuation and stop words, lemmatize.
   The cleaned text goes into a new `Text` column, the original `Text Description` is kept.

In [23]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


# ============================================================
# Utility: clean one text response
# ============================================================
def clean_text(text):
    if pd.isna(text):
        text = ""

    tokens = word_tokenize(str(text).lower())

    clean_tokens = []
    for token in tokens:
        token = token.translate(str.maketrans("", "", string.punctuation)).strip()

        if token == "" or token in stop_words:
            continue

        clean_tokens.append(lemmatizer.lemmatize(token))

    return " ".join(clean_tokens)

In [24]:
text_df = add_dataid(text_df)
print("TEXT raw shape:", text_df.shape)

text_df = drop_missing(text_df, text_features + identifiers, "TEXT")
text_df.insert(2, "Text", text_df["Text Description"].apply(clean_text))

print("TEXT clean shape:", text_df.shape)
display(text_df[["Data ID", "Text Description", "Text"]].head())

TEXT raw shape: (296, 14)
TEXT: 0 rows with missing values in 6 required columns
TEXT clean shape: (296, 15)


,Data ID,Text Description,Text
0,1001,In this video childrens were shown whoes homes...,video childrens shown whoes home destroyed war...
1,1002,In this video people and childrens were shown ...,video people childrens shown slum area video e...
2,1003,In this video a person who was confined in jai...,video person confined jail run aways jail nt f...
3,1004,This video was about war and it impact all nat...,video war impact nation equally video scene wa...
4,1005,In this video I jumped into a canyon using a r...,video jumped canyon using rope felt little sca...


## 9. Align modalities

Keep only the `Data ID`s present in all three modalities and sort by `Data ID`, so row *i* is the same
participant and video in every file. The multimodal notebooks rely on this.

In [25]:
common_ids = set(eda_df["Data ID"]) & set(ppg_df["Data ID"]) & set(text_df["Data ID"])

aligned = {}
for name, df in [("EDA", eda_df), ("PPG", ppg_df), ("TEXT", text_df)]:
    dropped = len(df) - df["Data ID"].isin(common_ids).sum()
    print(f"{name}: dropping {dropped} rows not present in all modalities")

    aligned[name] = (
        df[df["Data ID"].isin(common_ids)]
        .sort_values("Data ID")
        .reset_index(drop=True)
    )

eda_df, ppg_df, text_df = aligned["EDA"], aligned["PPG"], aligned["TEXT"]

# ============================================================
# Sanity checks: same rows and same labels in every modality
# ============================================================
assert eda_df["Data ID"].is_unique, "Duplicate Data ID in EDA"
assert eda_df["Data ID"].equals(ppg_df["Data ID"]), "EDA and PPG rows are not aligned"
assert eda_df["Data ID"].equals(text_df["Data ID"]), "EDA and TEXT rows are not aligned"

for col in ["arousal_category", "valence_category", "taskwiselabel"]:
    assert (eda_df[col].astype(int) == ppg_df[col].astype(int)).all(), f"{col} differs between EDA and PPG"
    assert (eda_df[col].astype(int) == text_df[col].astype(int)).all(), f"{col} differs between EDA and TEXT"

print("Aligned rows:", len(eda_df))
print("Participants:", eda_df["Participant ID"].nunique())

EDA: dropping 0 rows not present in all modalities
PPG: dropping 0 rows not present in all modalities
TEXT: dropping 0 rows not present in all modalities
Aligned rows: 296
Participants: 37


## 10. Save preprocessed files

In [26]:
eda_df.to_csv(OUTPUT_CONFIG["eda_path"], index=False)
ppg_df.to_csv(OUTPUT_CONFIG["ppg_path"], index=False)
text_df.to_csv(OUTPUT_CONFIG["text_path"], index=False)
text_df.to_csv(OUTPUT_CONFIG["text_labels_path"], index=False)

for key in ["eda_path", "ppg_path", "text_path", "text_labels_path"]:
    print("Saved:", OUTPUT_CONFIG[key])

Saved: EEVR/Data_files/EDA_preprocessed_aligned.csv
Saved: EEVR/Data_files/PPG_preprocessed_aligned.csv
Saved: EEVR/Data_files/TEXT_preprocessed_aligned.csv
Saved: EEVR/Data_files/text_data_labels.csv
